<a href="https://colab.research.google.com/github/Mitul-Marimuthu/deep-learning/blob/project1/project1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train = X_train.reshape(60000, 784)/ 255.0
X_test = X_test.reshape(10000, 784) / 255.0

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train: {y_train.shape}, Test: {y_test.shape}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Train: (60000, 784), Test: (10000, 784)
Train: (60000,), Test: (10000,)


In [3]:
# initialize weights
np.random.seed(42)

# small weights used because if weights start larger, activations
# saturate and gradients vanish before training ever begins
def init_params():
  # layer 1: 784 inputs --> 128 hidden neurons
  W1 = np.random.randn(784, 128) * 0.01 # small random weights
  b1 = np.zeros((1, 128))

  # Layer 2: 128 hidden -> 10 outputs -> one per digit class
  W2 = np.random.randn(128, 10) * 0.01
  b2 = np.zeros((1, 10))

  return W1, b1, W2, b2

In [4]:
# Forward pass

# gets the relu (rectified linear unit) value
# enables faster training and better gradient flow in
# deep neural networks, mitigating the vanishing gradient
# problem.
def relu(z):
  return np.maximum(0, z)

# turns raw scores into probabilites
def softmax(z):
  # subtract max for numerical stability (prevents overflow)
  z = z - np.max(z, axis=1, keepdims=True) # column max
  exp_z = np.exp(z)
  return exp_z / np.sum(exp_z, axis=1, keepdims=True)

def forward(X, W1, b1, W2, b2):
  # layer 1
  Z1 = X @ W1 + b1 # (batch, 784) @ (784, 128) = (batch, 128)
  A1 = relu(Z1) # apply relu activation

  # Layer 2
  Z2 = A1 @ W2 + b2 # (batch, 128) @ (128, 10) = (batch, 10)
  A2 = softmax(Z2)

  return Z1, A1, Z2, A2

In [5]:
# Loss function
# measures how wrong predictions were
def cross_entropy_loss(A2, y, batch_size):
  # one hot encode labels
  one_hot = np.zeros_like(A2)
  one_hot[np.arange(batch_size), y] = 1

  # cross entropy: -sum(true * log(predicted))
  log_probs = np.log(A2 + 1e-8) # 1e-8 avoids 0
  # highly penalizes confident wrong predications
  loss = -np.sum(one_hot * log_probs) / batch_size # averages loss
  # across all images

  return loss, one_hot

In [6]:
# sanity check for loss function
# Fake a batch of 3 images, 10 classes
A2 = np.array([
    [0.01, 0.01, 0.01, 0.90, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],  # confident, correct (3)
    [0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10, 0.10],  # totally uncertain
    [0.90, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],  # confident, WRONG (true=3)
])
y = np.array([3, 3, 3])

loss, _ = cross_entropy_loss(A2, y, batch_size=3)
print(f"Loss: {loss:.4f}")
# Confident correct → low, uncertain → medium, confident wrong → high

Loss: 2.3377


In [7]:
# backpropagation
# answers --> which weights are responsible, and in which direction
# should they move to reduce the loss

# computes gradients layer by layer using the chain rule
# dL/dW2 = dL/dA2 * dA2/dZ2 * dZ2/dW2

def backward(X, Z1, A1, A2, W2, one_hot, batch_size):
  dZ2 = A2 - one_hot # dradient of the loss with respect to
  # preactivation of the output layer
  # every wrong class has a positive gradient

  # gradient of loss with respect to W2
  # EVERY GRADIENT MUST BE THE SAME SHAPE OF THE GRADIENT
  # IT IS OF
  dW2 = A1.T @ dZ2 / batch_size
  db2 = np.sum(dZ2, axis=0, keepdims=True) / batch_size

  # Hidden layer
  dA1 = dZ2 @ W2.T
  dZ1 = dA1 * (Z1 > 0)
  dW1 = X.T @ dZ1 / batch_size
  db1 = np.sum(dZ1, axis=0, keepdims=True) / batch_size

  return dW1, db1, dW2, db2

  # Loss
  # ↓  dZ2 = A2 - one_hot          (error at output)
  # ↓  dW2 = A1.T @ dZ2            (how much did W2 cause this?)
  # ↓  dA1 = dZ2 @ W2.T            (propagate error back through W2)
  # ↓  dZ1 = dA1 * (Z1 > 0)        (block gradient through dead ReLUs)
  # ↓  dW1 = X.T @ dZ1             (how much did W1 cause this?)

In [16]:
def gradient_check(X, y, W1, b1, W2, b2, i=203, j=64, epsilon=1e-5):
    # Compute analytical gradient for one weight
    Z1, A1, Z2, A2 = forward(X, W1, b1, W2, b2)
    loss, one_hot = cross_entropy_loss(A2, y, X.shape[0])
    dW1, db1, dW2, db2 = backward(X, Z1, A1, A2, W2, one_hot, X.shape[0])

    # Numerically estimate the gradient for W1[0,0]
    W1_plus = W1.copy(); W1_plus[i,j] += epsilon
    _, _, _, A2_plus = forward(X, W1_plus, b1, W2, b2)
    loss_plus, _ = cross_entropy_loss(A2_plus, y, X.shape[0])

    W1_minus = W1.copy(); W1_minus[i,j] -= epsilon
    _, _, _, A2_minus = forward(X, W1_minus, b1, W2, b2)
    loss_minus, _ = cross_entropy_loss(A2_minus, y, X.shape[0])

    numerical_grad = (loss_plus - loss_minus) / (2 * epsilon)
    analytical_grad = dW1[i, j]

    print(f"Numerical:  {numerical_grad:.8f}")
    print(f"Analytical: {analytical_grad:.8f}")
    print(f"Match: {np.isclose(numerical_grad, analytical_grad, rtol=1e-4)}")

# Use weights that have already been initialized, not fresh ones
W1, b1, W2, b2 = init_params()

# Scale up weights so neurons actually fire strongly
W1 *= 10
W2 *= 10

gradient_check(X_train[100:105], y_train[100:105], W1, b1, W2, b2, i=203, j=64)

Numerical:  0.00000000
Analytical: 0.00000000
Match: True


In [14]:
W1, b1, W2, b2 = init_params()
W1 *= 10
W2 *= 10

X_sample = X_train[100:105]
y_sample = y_train[100:105]

# Check if the pixels at position 14 are actually nonzero
print("Pixel values at row 14:", X_sample[:, 14])

# Check if neurons are actually firing
Z1, A1, Z2, A2 = forward(X_sample, W1, b1, W2, b2)
print("A1 (hidden activations) sample:", A1[0, :5])
print("Any neurons firing?", np.any(A1 > 0))

# Check the raw gradient before isolating one weight
_, _, _, A2_ = forward(X_sample, W1, b1, W2, b2)
loss, one_hot = cross_entropy_loss(A2_, y_sample, 5)
dW1, db1_g, dW2, db2_g = backward(X_sample, Z1, A1, A2_, W2, one_hot, 5)
print("dW1 max value:", np.max(np.abs(dW1)))
print("dW1[14,64]:", dW1[14, 64])

# insight - a weight connecting a 0 input to any neuron will alwyas
# have a zero gradient -- it literally cannot affect the output

Pixel values at row 14: [0. 0. 0. 0. 0.]
A1 (hidden activations) sample: [0.         1.95627746 0.         0.         0.        ]
Any neurons firing? True
dW1 max value: 0.10986404609907055
dW1[14,64]: 0.0
